# Module 03 — Unsupervised & Statistical Learning
## 03-08: Kernel Methods & Feature Maps

**Objective:** Implement the kernel trick from first principles — from explicit feature maps through Mercer's theorem, kernel PCA, kernel density estimation, and random Fourier feature approximations — and reveal the connection between softmax attention and kernel functions.

**Prerequisites:** 1-06 (Linear Algebra), 2-05 (SVMs — introduced the kernel trick)

## Part 0 — Setup & Prerequisites

This notebook owns **kernel PCA**, **kernel density estimation**, **kernel methods**, and **random Fourier features**. The kernel trick itself was introduced in 2-05 (SVMs); we build on that foundation here without re-teaching it.

**Prerequisites:** 1-06 (Linear Algebra), 2-05 (SVMs — kernel trick introduction)

In [ ]:
import sys
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.special import logsumexp
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.decomposition import KernelPCA as SklearnKernelPCA
from sklearn.model_selection import (
    cross_val_score,
    cross_val_score as _cv_score,
    train_test_split as tts,
)
from sklearn.neighbors import KernelDensity as SklearnKDE
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Experiment constants ──────────────────────────────────────────────────────
N_SAMPLES    = 400
N_COMPONENTS = 2       # kernel PCA output dimensions
GAMMA_RBF    = 1.0    # RBF kernel bandwidth parameter
DEGREE_POLY  = 3      # polynomial kernel degree
N_RFF        = 500    # Random Fourier Features count
KDE_BW       = None   # None = Silverman's rule

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
COLORS = list(plt.cm.tab10.colors)
print('Configuration loaded.')

### Data Loading & EDA

We generate three non-linearly separable datasets to showcase kernels:
- **Moons**: two interleaved half-circles — linearly inseparable
- **Circles**: concentric rings — radially symmetric, ideal for RBF kernel
- **Blobs**: three Gaussian clusters — baseline comparison

In [ ]:
# ── Generate datasets ─────────────────────────────────────────────────────────
X_moons, y_moons   = make_moons(n_samples=N_SAMPLES, noise=0.1, random_state=SEED)
X_circles, y_circles = make_circles(n_samples=N_SAMPLES, noise=0.05,
                                     factor=0.4, random_state=SEED)
X_blobs, y_blobs   = make_blobs(n_samples=N_SAMPLES, centers=3,
                                 cluster_std=0.8, random_state=SEED)

# Standardise all datasets
scaler_m = StandardScaler()
scaler_c = StandardScaler()
scaler_b = StandardScaler()
X_moons   = scaler_m.fit_transform(X_moons)
X_circles = scaler_c.fit_transform(X_circles)
X_blobs   = scaler_b.fit_transform(X_blobs)

print('Dataset shapes:')
print(f'  Moons   : {X_moons.shape},   classes: {np.unique(y_moons)}')
print(f'  Circles : {X_circles.shape}, classes: {np.unique(y_circles)}')
print(f'  Blobs   : {X_blobs.shape},   classes: {np.unique(y_blobs)}')

# ── Scatter EDA ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
datasets = [(X_moons, y_moons, 'Moons'), (X_circles, y_circles, 'Circles'),
            (X_blobs,  y_blobs,  'Blobs')]

for ax, (X, y, name) in zip(axes, datasets):
    for cls in np.unique(y):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=[COLORS[cls]], alpha=0.6,
                   s=20, label=f'Class {cls}')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=9)

fig.suptitle('Non-Linearly Separable Benchmark Datasets', fontsize=14,
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── Linear separability check (PCA first PC variance explained) ───────────────
for X, y, name in datasets:
    u, s, _ = np.linalg.svd(X - X.mean(axis=0), full_matrices=False)
    var_exp = s**2 / (s**2).sum()
    print(f'{name}: PCA variance explained: PC1={var_exp[0]:.3f}, PC2={var_exp[1]:.3f}')

---
## Part 1 — Kernels, Feature Maps & Mercer's Theorem from Scratch

### 1.1 The Kernel Trick (reference: 2-05 SVMs)

A kernel $k(\mathbf{x}, \mathbf{z})$ implicitly computes an inner product in a (possibly infinite-dimensional) feature space $\mathcal{H}$:
$$k(\mathbf{x}, \mathbf{z}) = \langle \varphi(\mathbf{x}),\, \varphi(\mathbf{z}) \rangle_{\mathcal{H}}$$

Common kernels:
| Kernel | Formula | Feature Space |
|--------|---------|---------------|
| Linear | $\mathbf{x}^\top\mathbf{z}$ | $\mathbb{R}^d$ (identity map) |
| Polynomial | $(\gamma\,\mathbf{x}^\top\mathbf{z} + c)^p$ | $\mathbb{R}^{\binom{d+p}{p}}$ |
| RBF (Gaussian) | $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$ | Infinite-dimensional |
| Sigmoid | $\tanh(\alpha\,\mathbf{x}^\top\mathbf{z} + c)$ | (not always valid) |

### 1.2 Mercer's Theorem

A symmetric function $k(\mathbf{x}, \mathbf{z})$ is a **valid kernel** if and only if the kernel matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ is **positive semi-definite (PSD)** for all datasets:
$$\mathbf{c}^\top K \mathbf{c} \geq 0 \quad \forall \mathbf{c} \in \mathbb{R}^n$$

Equivalently: all eigenvalues of $K$ are non-negative.

In [ ]:
# ── Kernel functions ──────────────────────────────────────────────────────────

def linear_kernel(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    '''Linear kernel: k(x, z) = x^T z.

    Args:
        X1: Array of shape (n1, d).
        X2: Array of shape (n2, d).

    Returns:
        Kernel matrix of shape (n1, n2).
    '''
    return X1 @ X2.T


def polynomial_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    degree: int = DEGREE_POLY,
    coef0: float = 1.0,
    gamma: float = 1.0,
) -> np.ndarray:
    '''Polynomial kernel: k(x, z) = (gamma * x^T z + coef0)^degree.

    Args:
        X1:     Array of shape (n1, d).
        X2:     Array of shape (n2, d).
        degree: Polynomial degree.
        coef0:  Bias term (inhomogeneous if > 0).
        gamma:  Scaling of the inner product.

    Returns:
        Kernel matrix of shape (n1, n2).
    '''
    return (gamma * X1 @ X2.T + coef0) ** degree


def rbf_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    gamma: float = GAMMA_RBF,
) -> np.ndarray:
    '''Radial Basis Function (Gaussian) kernel: k(x, z) = exp(-gamma ||x-z||^2).

    Implemented via the identity ||x-z||^2 = ||x||^2 + ||z||^2 - 2 x^T z
    to avoid materialising the (n1, n2, d) difference tensor.

    Args:
        X1:    Array of shape (n1, d).
        X2:    Array of shape (n2, d).
        gamma: Bandwidth parameter (larger = narrower kernel).

    Returns:
        Kernel matrix of shape (n1, n2).
    '''
    sq1  = np.sum(X1 ** 2, axis=1, keepdims=True)  # (n1, 1)
    sq2  = np.sum(X2 ** 2, axis=1, keepdims=True)  # (n2, 1)
    dist = sq1 + sq2.T - 2.0 * (X1 @ X2.T)        # (n1, n2)
    return np.exp(-gamma * np.maximum(dist, 0.0))  # clamp numerics


def sigmoid_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    alpha: float = 0.01,
    coef0: float = 0.0,
) -> np.ndarray:
    '''Sigmoid kernel: k(x, z) = tanh(alpha * x^T z + coef0).

    Note: valid kernel (PSD) only for specific alpha, coef0 ranges.

    Args:
        X1:    Array of shape (n1, d).
        X2:    Array of shape (n2, d).
        alpha: Slope scaling.
        coef0: Bias.

    Returns:
        Kernel matrix of shape (n1, n2).
    '''
    return np.tanh(alpha * X1 @ X2.T + coef0)


def compute_kernel_matrix(
    X1: np.ndarray,
    X2: np.ndarray,
    kernel: str = 'rbf',
    **kwargs,
) -> np.ndarray:
    '''Compute a kernel matrix using a named kernel function.

    Args:
        X1:     Array of shape (n1, d).
        X2:     Array of shape (n2, d).
        kernel: One of 'linear', 'poly', 'rbf', 'sigmoid'.
        **kwargs: Passed to the chosen kernel function.

    Returns:
        Kernel matrix of shape (n1, n2).
    '''
    fns = {
        'linear':  linear_kernel,
        'poly':    polynomial_kernel,
        'rbf':     rbf_kernel,
        'sigmoid': sigmoid_kernel,
    }
    if kernel not in fns:
        raise ValueError(f'Unknown kernel: {kernel!r}. Choose from {list(fns)}')
    return fns[kernel](X1, X2, **kwargs)


# ── Quick kernel evaluation demo ──────────────────────────────────────────────
X_demo = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
print('Demo kernel values (3x3 matrix, X_demo rows):')
for name in ['linear', 'rbf', 'poly']:
    K = compute_kernel_matrix(X_demo, X_demo, kernel=name)
    print(f'  {name}: diag={np.round(np.diag(K), 3)}, min_eigenvalue={np.linalg.eigvalsh(K).min():.6f}')

### 1.3 Explicit Feature Map for the Polynomial Kernel

For degree 2 in 2D, we can write the feature map explicitly. If $\mathbf{x} = [x_1, x_2]$:
$$\varphi(\mathbf{x}) = [x_1^2,\; \sqrt{2}\,x_1 x_2,\; x_2^2]$$

Then $k(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^\top \mathbf{z})^2 = \varphi(\mathbf{x})^\top \varphi(\mathbf{z})$.

For degree 3 and higher, or for RBF (where the feature space is infinite-dimensional), the explicit map is intractable — this is exactly why the kernel trick is powerful.

In [ ]:
def poly2_feature_map(X: np.ndarray) -> np.ndarray:
    '''Explicit degree-2 homogeneous polynomial feature map for 2-D input.

    Computes phi(x) = [x1^2, sqrt(2)*x1*x2, x2^2] such that
    phi(x)^T phi(z) = (x^T z)^2.

    Args:
        X: Input data of shape (n, 2).

    Returns:
        Feature matrix of shape (n, 3).
    '''
    assert X.shape[1] == 2, 'poly2_feature_map requires 2-D input'
    x1 = X[:, 0:1]
    x2 = X[:, 1:2]
    return np.hstack([x1 ** 2, np.sqrt(2.0) * x1 * x2, x2 ** 2])


def poly3_feature_map(X: np.ndarray, coef0: float = 1.0) -> np.ndarray:
    '''Explicit degree-3 inhomogeneous feature map for 2-D input.

    For k(x,z) = (x^T z + coef0)^3, the feature vector has 10 components
    corresponding to all monomials of degree <= 3 in x1, x2.

    Args:
        X:      Input data of shape (n, 2).
        coef0:  Constant offset c in the kernel (x^T z + c)^3.

    Returns:
        Feature matrix of shape (n, 10).
    '''
    x1 = X[:, 0]
    x2 = X[:, 1]
    c  = coef0
    # Monomials of (x1, x2, sqrt(c)) up to degree 3 via trinomial theorem
    s  = np.sqrt(c)
    phi = np.column_stack([
        x1**3, x2**3, s**3 * np.ones(len(x1)),
        np.sqrt(3.0) * x1**2 * x2,
        np.sqrt(3.0) * x1 * x2**2,
        np.sqrt(3.0) * x1**2 * s,
        np.sqrt(3.0) * x2**2 * s,
        np.sqrt(3.0) * x1 * s**2,
        np.sqrt(3.0) * x2 * s**2,
        np.sqrt(6.0) * x1 * x2 * s,
    ])
    return phi


# ── Verify: explicit map matches kernel evaluation ────────────────────────────
X_v = np.array([[2.0, 1.0], [0.5, 3.0], [-1.0, 0.5]])

# Degree-2 homogeneous
phi2      = poly2_feature_map(X_v)
K_via_map = phi2 @ phi2.T
K_direct  = polynomial_kernel(X_v, X_v, degree=2, coef0=0.0, gamma=1.0)
max_err2  = np.max(np.abs(K_via_map - K_direct))
print(f'Poly-2 feature map vs kernel: max abs error = {max_err2:.2e}')
assert max_err2 < 1e-10

# Degree-3 with coef0=1
phi3      = poly3_feature_map(X_v, coef0=1.0)
K_via_map3 = phi3 @ phi3.T
K_direct3  = polynomial_kernel(X_v, X_v, degree=3, coef0=1.0, gamma=1.0)
max_err3   = np.max(np.abs(K_via_map3 - K_direct3))
print(f'Poly-3 feature map vs kernel: max abs error = {max_err3:.2e}')
assert max_err3 < 1e-8
print('Both explicit feature maps match kernel evaluations.')

# ── Feature space dimensionality ──────────────────────────────────────────────
for d in [2, 5, 10, 50, 100]:
    from math import comb
    n_feats_deg3 = comb(d + 3, 3)
    print(f'  d={d:3d}: poly-degree-3 feature space dim = {n_feats_deg3}')
print('  d=any: RBF feature space dim = inf (Hermite basis)')

### 1.4 Mercer's Theorem in Practice

A kernel is valid if its Gram matrix is PSD for any dataset. We verify this empirically: RBF and polynomial kernels always have non-negative eigenvalues, while the sigmoid kernel can violate PSD for some parameter choices.

In [ ]:
def validate_mercer(
    K: np.ndarray,
    name: str = 'kernel',
    tol: float = 1e-8,
) -> dict:
    '''Validate whether a kernel matrix is positive semi-definite.

    Args:
        K:    Square symmetric kernel matrix of shape (n, n).
        name: Display name for the kernel.
        tol:  Tolerance for considering an eigenvalue non-negative.

    Returns:
        Dict with keys: name, min_eigenvalue, is_psd, n_negative.
    '''
    eigenvalues  = np.linalg.eigvalsh(K)
    min_eig      = float(eigenvalues.min())
    n_negative   = int((eigenvalues < -tol).sum())
    is_psd       = n_negative == 0
    status       = 'PSD (valid)' if is_psd else f'NOT PSD ({n_negative} negative eigenvalues)'
    print(f'  {name:30s}: min_eig={min_eig:.6f},  {status}')
    return {'name': name, 'min_eigenvalue': min_eig, 'is_psd': is_psd,
            'n_negative': n_negative, 'eigenvalues': eigenvalues}


# ── Validate all kernels on the Moons dataset ─────────────────────────────────
print('Mercer validation on Moons dataset (n=400):')
X_val = X_moons[:50]   # 50-point subset for tractability
mercer_results = []
mercer_results.append(validate_mercer(linear_kernel(X_val, X_val),            'Linear'))
mercer_results.append(validate_mercer(rbf_kernel(X_val, X_val, gamma=1.0),    'RBF (gamma=1.0)'))
mercer_results.append(validate_mercer(rbf_kernel(X_val, X_val, gamma=10.0),   'RBF (gamma=10.0)'))
mercer_results.append(validate_mercer(polynomial_kernel(X_val, X_val, degree=2),   'Poly (degree=2)'))
mercer_results.append(validate_mercer(polynomial_kernel(X_val, X_val, degree=3),   'Poly (degree=3)'))
mercer_results.append(validate_mercer(sigmoid_kernel(X_val, X_val, alpha=0.01),    'Sigmoid (alpha=0.01)'))
mercer_results.append(validate_mercer(sigmoid_kernel(X_val, X_val, alpha=1.0),     'Sigmoid (alpha=1.0)'))

# ── Kernel matrix heatmaps ────────────────────────────────────────────────────
kernels_to_plot = [
    ('Linear',    linear_kernel(X_val, X_val)),
    ('RBF g=1',   rbf_kernel(X_val, X_val, gamma=1.0)),
    ('Poly d=3',  polynomial_kernel(X_val, X_val, degree=3)),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (kname, K) in zip(axes, kernels_to_plot):
    # Sort rows/cols by label for visual structure
    order = np.argsort(y_moons[:50])
    K_sorted = K[order][:, order]
    im = ax.imshow(K_sorted, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{kname} kernel matrix\n(rows/cols sorted by class)', fontsize=11)
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Sample index')
fig.suptitle('Kernel Matrices on Moons (n=50, sorted by class)', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

### 1.5 Kernel PCA: Mathematics

Kernel PCA applies PCA in the feature space $\mathcal{H}$ without explicitly computing $\varphi(\mathbf{x})$. The key steps:

**Step 1 — Kernel matrix:** Compute $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$.

**Step 2 — Centre in feature space:** Subtract the feature-space mean:
$$\tilde{K} = K - \mathbf{1}_n K - K \mathbf{1}_n + \mathbf{1}_n K \mathbf{1}_n, \quad (\mathbf{1}_n)_{ij} = 1/n$$

**Step 3 — Eigendecomposition:** Solve $\tilde{K} \boldsymbol{\alpha} = \lambda \boldsymbol{\alpha}$, normalise: $\boldsymbol{\alpha}_k \leftarrow \boldsymbol{\alpha}_k / \sqrt{\lambda_k}$.

**Step 4 — Project:** $z_i = \tilde{K}_i \boldsymbol{\alpha}$ where $\tilde{K}_i$ is the $i$-th row of the centred kernel matrix.

For new test points, centering uses the *training* row/column means.

In [ ]:
def center_kernel_matrix(
    K_train: np.ndarray,
) -> tuple:
    '''Centre the training kernel matrix in feature space.

    Implements: K_c = K - 1_n K - K 1_n + 1_n K 1_n
    where 1_n is the n x n matrix with all entries 1/n.

    Args:
        K_train: Symmetric kernel matrix of shape (n, n).

    Returns:
        K_c:         Centred kernel matrix of shape (n, n).
        col_mean:    Per-column mean vector of K_train, shape (n,).
        total_mean:  Grand mean of K_train (scalar).
    '''
    col_mean   = K_train.mean(axis=0)       # (n,)  mean of each column
    total_mean = float(col_mean.mean())     # scalar grand mean
    row_mean   = K_train.mean(axis=1)       # (n,)  mean of each row
    K_c = K_train - row_mean[:, None] - col_mean[None, :] + total_mean
    return K_c, col_mean, total_mean


def center_kernel_test(
    K_test: np.ndarray,
    col_mean_train: np.ndarray,
    total_mean_train: float,
) -> np.ndarray:
    '''Centre a test kernel matrix using training statistics.

    For each test point x_new and training point x_i:
    K_c[new, i] = K[new, i] - mean_i(K[new, .]) - col_mean_train[i] + total_mean_train

    Args:
        K_test:           Test kernel matrix of shape (n_test, n_train).
        col_mean_train:   Column means from training, shape (n_train,).
        total_mean_train: Grand mean from training (scalar).

    Returns:
        Centred test kernel matrix of shape (n_test, n_train).
    '''
    row_mean_test = K_test.mean(axis=1)                        # (n_test,)
    K_c = (K_test
           - row_mean_test[:, None]
           - col_mean_train[None, :]
           + total_mean_train)
    return K_c


def fit_kernel_pca(
    K_c: np.ndarray,
    n_components: int,
) -> tuple:
    '''Fit kernel PCA from a centred kernel matrix.

    Computes the top n_components eigenvectors and eigenvalues of K_c,
    then normalises each eigenvector by 1/sqrt(lambda_k).

    Args:
        K_c:          Centred kernel matrix of shape (n, n).
        n_components: Number of principal components to retain.

    Returns:
        alphas:  Normalised eigenvectors, shape (n, n_components).
        lambdas: Eigenvalues (descending), shape (n_components,).
    '''
    eigenvalues, eigenvectors = np.linalg.eigh(K_c)
    idx          = np.argsort(eigenvalues)[::-1]   # descending order
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    lambdas      = eigenvalues[:n_components]
    alphas       = eigenvectors[:, :n_components]
    alphas       = alphas / np.sqrt(np.maximum(lambdas, 1e-10))   # normalise
    return alphas, lambdas


def transform_kernel_pca(
    K_c_test: np.ndarray,
    alphas: np.ndarray,
) -> np.ndarray:
    '''Project test data using pre-fitted kernel PCA components.

    Args:
        K_c_test: Centred test kernel matrix of shape (n_test, n_train).
        alphas:   Normalised eigenvectors of shape (n_train, n_components).

    Returns:
        Projected coordinates of shape (n_test, n_components).
    '''
    return K_c_test @ alphas   # (n_test, n_components)

---
## Part 2 — Putting It All Together

We assemble the kernel PCA components into a clean class with the familiar `fit` / `transform` API, then add the KDE and RFF implementations.

In [ ]:
class KernelPCA:
    '''Kernel Principal Component Analysis — from-scratch implementation.

    Maps data into a low-dimensional space using the kernel trick,
    capturing non-linear structure that standard PCA misses.

    Attributes:
        n_components_:           Number of components fitted.
        kernel_:                 Kernel name used.
        X_fit_:                  Training data, shape (n, d).
        alphas_:                 Normalised eigenvectors, shape (n, n_components).
        lambdas_:                Eigenvalues, shape (n_components,).
        col_mean_:               Training kernel column means, shape (n,).
        total_mean_:             Training kernel grand mean (float).
        explained_variance_ratio_: Fraction of variance per component.
    '''

    def __init__(
        self,
        n_components: int = N_COMPONENTS,
        kernel: str = 'rbf',
        gamma: float = GAMMA_RBF,
        degree: int = DEGREE_POLY,
        coef0: float = 1.0,
        alpha_reg: float = 1e-6,
    ) -> None:
        '''Store hyperparameters.

        Args:
            n_components: Number of components to compute.
            kernel:       'rbf', 'poly', 'linear', or 'sigmoid'.
            gamma:        Bandwidth for RBF; scaling for poly/sigmoid.
            degree:       Degree for polynomial kernel.
            coef0:        Offset for polynomial/sigmoid kernel.
            alpha_reg:    Regularisation added to eigenvalues.
        '''
        self.n_components = n_components
        self.kernel       = kernel
        self.gamma        = gamma
        self.degree       = degree
        self.coef0        = coef0
        self.alpha_reg    = alpha_reg
        self.X_fit_                   = None
        self.alphas_                  = None
        self.lambdas_                 = None
        self.col_mean_                = None
        self.total_mean_              = None
        self.explained_variance_ratio_ = None

    def _kernel(self, X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
        '''Compute the kernel matrix between X1 and X2.

        Args:
            X1: Array of shape (n1, d).
            X2: Array of shape (n2, d).

        Returns:
            Kernel matrix of shape (n1, n2).
        '''
        if self.kernel == 'rbf':
            return rbf_kernel(X1, X2, gamma=self.gamma)
        if self.kernel == 'poly':
            return polynomial_kernel(X1, X2, degree=self.degree,
                                     coef0=self.coef0, gamma=self.gamma)
        if self.kernel == 'linear':
            return linear_kernel(X1, X2)
        if self.kernel == 'sigmoid':
            return sigmoid_kernel(X1, X2, alpha=self.gamma, coef0=self.coef0)
        raise ValueError(f'Unknown kernel: {self.kernel!r}')

    def fit(self, X: np.ndarray) -> 'KernelPCA':
        '''Fit kernel PCA on training data.

        Args:
            X: Training data of shape (n, d).

        Returns:
            self — fitted estimator.
        '''
        self.X_fit_  = X.copy()
        K_train      = self._kernel(X, X)
        K_c, self.col_mean_, self.total_mean_ = center_kernel_matrix(K_train)
        self.alphas_, self.lambdas_ = fit_kernel_pca(K_c, self.n_components)
        # Explained variance ratio (only count positive eigenvalues)
        K_full_eigs = np.linalg.eigvalsh(K_c)
        total_var   = float(np.sum(np.maximum(K_full_eigs, 0.0)))
        self.explained_variance_ratio_ = (
            self.lambdas_ / (total_var + 1e-10)
        )
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Project X into the kernel PCA space.

        Args:
            X: Data of shape (n, d).

        Returns:
            Projected coordinates of shape (n, n_components).
        '''
        K_test = self._kernel(X, self.X_fit_)
        K_c    = center_kernel_test(K_test, self.col_mean_, self.total_mean_)
        return transform_kernel_pca(K_c, self.alphas_)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit and project in one step.

        Args:
            X: Training data of shape (n, d).

        Returns:
            Projected coordinates of shape (n, n_components).
        '''
        return self.fit(X).transform(X)

### Sanity Check — vs sklearn KernelPCA

We verify our implementation matches sklearn's `KernelPCA` on both Moons and Circles. The projected coordinates may differ by a sign flip per component (eigenvector sign is ambiguous), so we check absolute correlation.

In [ ]:
# ── Sanity check: our KernelPCA vs sklearn on Moons ──────────────────────────
for dataset_name, X_d, y_d in [('Moons', X_moons, y_moons),
                                 ('Circles', X_circles, y_circles)]:
    # Our implementation
    kpca_ours = KernelPCA(n_components=2, kernel='rbf', gamma=GAMMA_RBF)
    Z_ours    = kpca_ours.fit_transform(X_d)

    # sklearn reference
    kpca_skl  = SklearnKernelPCA(n_components=2, kernel='rbf', gamma=GAMMA_RBF)
    Z_skl     = kpca_skl.fit_transform(X_d)

    # Compare: correlation per component (abs, sign-invariant)
    corrs = []
    for c in range(2):
        corr = float(np.corrcoef(Z_ours[:, c], Z_skl[:, c])[0, 1])
        corrs.append(abs(corr))
    print(f'{dataset_name}: component correlations vs sklearn = '
          f'PC1={corrs[0]:.6f}, PC2={corrs[1]:.6f}')
    assert min(corrs) > 0.999, 'KernelPCA deviates from sklearn!'

print('Sanity check passed: our KernelPCA matches sklearn.')

# ── Visualise KernelPCA projections ──────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
row_data  = [(X_moons, y_moons, 'Moons'), (X_circles, y_circles, 'Circles')]
kernels   = [('linear', {}), ('rbf', {'gamma': 1.0}), ('poly', {'degree': 3})]

for row_idx, (X_d, y_d, dsname) in enumerate(row_data):
    # Original data (leftmost column)
    ax = axes[row_idx, 0]
    for cls in np.unique(y_d):
        mask = y_d == cls
        ax.scatter(X_d[mask, 0], X_d[mask, 1], c=[COLORS[cls]], alpha=0.6,
                   s=15, label=f'C{cls}')
    ax.set_title(f'{dsname} — Original', fontsize=10)
    ax.legend(fontsize=8)

    # Kernel PCA projections for each kernel
    for col_idx, (kname, kargs) in enumerate(kernels):
        ax = axes[row_idx, col_idx + 1]
        kpca = KernelPCA(n_components=2, kernel=kname, **kargs)
        Z    = kpca.fit_transform(X_d)
        for cls in np.unique(y_d):
            mask = y_d == cls
            ax.scatter(Z[mask, 0], Z[mask, 1], c=[COLORS[cls]], alpha=0.6,
                       s=15)
        evr = kpca.explained_variance_ratio_
        ax.set_title(f'{dsname} — {kname} kPCA\nEVR: {evr[0]:.2f}/{evr[1]:.2f}',
                     fontsize=9)
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')

fig.suptitle('Kernel PCA: Original vs Projected (2 datasets x 3 kernels)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Kernel Density Estimation (KDE)

KDE places a kernel at each training point and sums to estimate a smooth density:
$$\hat{p}(\mathbf{x}) = \frac{1}{n\,h^d}\sum_{i=1}^{n} K\!\left(\frac{\mathbf{x}-\mathbf{x}_i}{h}\right)$$

With the Gaussian kernel: $K(u) = (2\pi)^{-d/2} \exp(-\|u\|^2/2)$, giving bandwidth $h$ control over smoothness. We implement cross-validation bandwidth selection.

In [ ]:
def silverman_bandwidth(X: np.ndarray) -> float:
    '''Silverman rule-of-thumb bandwidth: h = sigma_bar * n^{-1/(d+4)}.

    Args:
        X: Training data of shape (n, d).

    Returns:
        Bandwidth scalar h.
    '''
    n, d  = X.shape
    sigma = float(np.mean(X.std(axis=0)))
    return sigma * (n ** (-1.0 / (d + 4.0)))


def gaussian_kde_log_density(
    X_train: np.ndarray,
    X_eval: np.ndarray,
    bandwidth: float,
) -> np.ndarray:
    '''Evaluate multivariate Gaussian KDE log-density at evaluation points.

    Uses log-sum-exp over training kernels for numerical stability.
    Each kernel is isotropic Gaussian with bandwidth h (scalar).

    Args:
        X_train:   Training data (kernel centres) of shape (n_train, d).
        X_eval:    Points to evaluate, shape (n_eval, d).
        bandwidth: Kernel bandwidth h (standard deviation).

    Returns:
        Log-density values of shape (n_eval,).
    '''
    n_train, d = X_train.shape
    h          = bandwidth
    # Unnormalised log-kernel: -0.5 ||x - xi||^2 / h^2
    diffs  = X_eval[:, None] - X_train[None]          # (n_eval, n_train, d)
    sq_d   = np.sum(diffs ** 2, axis=2) / (h ** 2)    # (n_eval, n_train)
    log_k  = -0.5 * sq_d                              # (n_eval, n_train)
    # Log-sum-exp + normalisation constant
    log_norm = -d * np.log(h) - 0.5 * d * np.log(2.0 * np.pi)
    log_dens = logsumexp(log_k, axis=1) + log_norm - np.log(n_train)
    return log_dens


def kde_bandwidth_loo_cv(
    X: np.ndarray,
    bandwidths: np.ndarray,
    max_loo: int = 200,
) -> np.ndarray:
    '''Select KDE bandwidth via leave-one-out log-likelihood.

    Evaluates each bandwidth on a random subset of up to max_loo points
    using leave-one-out cross-validation.

    Args:
        X:          Training data of shape (n, d).
        bandwidths: Array of bandwidth values to evaluate.
        max_loo:    Maximum number of LOO evaluations (subset for speed).

    Returns:
        Array of LOO log-likelihoods, shape (len(bandwidths),).
    '''
    rng    = np.random.RandomState(SEED)
    n      = X.shape[0]
    idx    = rng.choice(n, size=min(max_loo, n), replace=False)
    X_sub  = X[idx]
    n_sub  = X_sub.shape[0]
    loo_lls = np.zeros(len(bandwidths))

    for bi, h in enumerate(bandwidths):
        ll = 0.0
        for i in range(n_sub):
            # LOO: exclude point i from the training set
            X_loo = np.delete(X_sub, i, axis=0)
            x_i   = X_sub[i:i + 1]
            ld    = gaussian_kde_log_density(X_loo, x_i, h)
            ll   += float(ld[0])
        loo_lls[bi] = ll
    return loo_lls


# ── Bandwidth selection demo ──────────────────────────────────────────────────
bandwidths_to_try = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75, 1.0])
h_silverman       = silverman_bandwidth(X_moons)
print(f'Silverman bandwidth for Moons: {h_silverman:.4f}')

loo_lls = kde_bandwidth_loo_cv(X_moons, bandwidths_to_try, max_loo=100)
best_idx = int(np.argmax(loo_lls))
best_h   = float(bandwidths_to_try[best_idx])
print(f'Best bandwidth (LOO-CV): {best_h:.3f}')

df_bw = pd.DataFrame({'bandwidth': bandwidths_to_try, 'LOO log-lik': np.round(loo_lls, 2)})
print(df_bw.set_index('bandwidth').to_string())

# ── KDE density heatmap for Moons ─────────────────────────────────────────────
resolution  = 150
x1_r = np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, resolution)
x2_r = np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, resolution)
xx1, xx2    = np.meshgrid(x1_r, x2_r)
X_grid      = np.column_stack([xx1.ravel(), xx2.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h, title in zip(
    axes,
    [0.1, h_silverman, 1.0],
    ['Undersmoothed h=0.1', f'Silverman h={h_silverman:.2f}', 'Oversmoothed h=1.0'],
):
    log_dens = gaussian_kde_log_density(X_moons, X_grid, bandwidth=h)
    ax.contourf(xx1, xx2, log_dens.reshape(xx1.shape), levels=25,
                cmap='viridis', alpha=0.85)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c='white', alpha=0.4, s=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('F1')
axes[0].set_ylabel('F2')
fig.suptitle('KDE Log-Density: Bandwidth Effect (Moons)', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

### KDE Bandwidth Selection: LOO-CV Curve

Bandwidth `h` controls the bias–variance tradeoff in KDE. Small `h` → spiky, overfit density; large `h` → over-smoothed density. We select `h` by maximising the leave-one-out (LOO) log-likelihood and compare against the closed-form **Silverman rule-of-thumb**.

In [ ]:
# ── Bandwidth selection: LOO-CV score curve ────────────────────────────────
print("=== KDE Bandwidth Selection via Leave-One-Out CV ===\n")

X_moon_kde, _ = make_moons(n_samples=200, noise=0.15, random_state=SEED)
X_moon_kde = StandardScaler().fit_transform(X_moon_kde)

# Fine bandwidth grid on log scale
bandwidths_fine: np.ndarray = np.logspace(-2, 0.5, 50)

# Our LOO-CV implementation
loo_scores: np.ndarray = kde_bandwidth_loo_cv(X_moon_kde, bandwidths_fine, max_loo=200)

# Sklearn 5-fold CV as a cross-check
sk_scores: list = []
for h_val in bandwidths_fine:
    kde_sk = SklearnKDE(bandwidth=h_val, kernel='gaussian')
    sc_val = cross_val_score(kde_sk, X_moon_kde, cv=5).mean()
    sk_scores.append(sc_val)
sk_scores_arr: np.ndarray = np.array(sk_scores)

best_h_ours: float = float(bandwidths_fine[np.nanargmax(loo_scores)])
best_h_sklearn: float = float(bandwidths_fine[np.argmax(sk_scores_arr)])
silverman_h: float = float(silverman_bandwidth(X_moon_kde))

print(f'Our LOO-CV best bandwidth   : {best_h_ours:.4f}')
print(f'Sklearn 5-fold CV best      : {best_h_sklearn:.4f}')
print(f'Silverman rule bandwidth    : {silverman_h:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.semilogx(bandwidths_fine, loo_scores, 'b-', lw=2, label='Our LOO-CV')
ax.semilogx(bandwidths_fine, sk_scores_arr, 'r--', lw=2, label='Sklearn 5-fold CV')
ax.axvline(best_h_ours, color='b', ls=':', alpha=0.7, label=f'Best (ours) h={best_h_ours:.3f}')
ax.axvline(best_h_sklearn, color='r', ls=':', alpha=0.7, label=f'Best (sk) h={best_h_sklearn:.3f}')
ax.axvline(silverman_h, color='g', ls='--', alpha=0.7, label=f'Silverman h={silverman_h:.3f}')
ax.set_xlabel('Bandwidth h (log scale)')
ax.set_ylabel('Mean log-likelihood')
ax.set_title('Bandwidth Selection Curve')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Density contours for four representative bandwidths
ax = axes[1]
xx_kde, yy_kde = np.meshgrid(np.linspace(-3, 4, 70), np.linspace(-2, 3, 70))
grid_kde = np.column_stack([xx_kde.ravel(), yy_kde.ravel()])
bandwidth_labels: list = [
    (0.05,          'red',    'h=0.05 undersmooth'),
    (best_h_ours,   'blue',   f'h={best_h_ours:.3f} LOO-CV'),
    (silverman_h,   'green',  f'h={silverman_h:.3f} Silverman'),
    (0.80,          'orange', 'h=0.80 oversmooth'),
]
for bw, col, lbl in bandwidth_labels:
    log_d = gaussian_kde_log_density(X_moon_kde, grid_kde, bw)
    ax.contour(xx_kde, yy_kde, log_d.reshape(xx_kde.shape),
               levels=5, colors=[col], alpha=0.65, linewidths=1.3)
    ax.plot([], [], color=col, label=lbl, lw=2)
ax.scatter(X_moon_kde[:, 0], X_moon_kde[:, 1], s=10, alpha=0.35, c='k', zorder=3)
ax.set_title('KDE Density Contours at 4 Bandwidths')
ax.legend(fontsize=8)
ax.set_xlim(-3, 4)
ax.set_ylim(-2, 3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kde_bandwidth_selection.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nConclusion: LOO-CV adapts to the true data shape; Silverman assumes normality.')
print('Under-smoothing (small h) captures noise; over-smoothing erases structure.')

### Random Fourier Features (RFF)

Bochner's theorem states that any shift-invariant PSD kernel has a Fourier transform $p(\boldsymbol{\omega}) \geq 0$:
$$k(\mathbf{x}, \mathbf{z}) = \int p(\boldsymbol{\omega}) e^{j\boldsymbol{\omega}^\top(\mathbf{x}-\mathbf{z})}\,d\boldsymbol{\omega}$$

For the RBF kernel $k(\mathbf{x},\mathbf{z}) = e^{-\gamma\|\mathbf{x}-\mathbf{z}\|^2}$, the spectral distribution is $p(\boldsymbol{\omega}) = \mathcal{N}(0, 2\gamma I)$. Sampling $D$ frequencies and using the cosine approximation:
$$z_j(\mathbf{x}) = \sqrt{\tfrac{2}{D}}\cos(\boldsymbol{\omega}_j^\top\mathbf{x} + b_j), \quad \boldsymbol{\omega}_j \sim \mathcal{N}(0, 2\gamma I),\; b_j \sim \text{Uniform}[0, 2\pi]$$

Then $k(\mathbf{x}, \mathbf{z}) \approx z(\mathbf{x})^\top z(\mathbf{z})$, converting kernel methods from $O(n^2)$ to $O(nD)$.

In [ ]:
def sample_rff_weights(
    d: int,
    D: int,
    gamma: float = GAMMA_RBF,
    random_state: int = SEED,
) -> tuple:
    '''Sample Random Fourier Feature weights for the RBF kernel.

    Draws spectral frequencies omega ~ N(0, 2*gamma*I) and
    phase offsets b ~ Uniform[0, 2*pi] as prescribed by Bochner's theorem.

    Args:
        d:            Input dimensionality.
        D:            Number of random features.
        gamma:        RBF kernel bandwidth parameter.
        random_state: Random seed.

    Returns:
        omega: Frequency matrix of shape (d, D).
        b:     Phase offset vector of shape (D,).
    '''
    rng   = np.random.RandomState(random_state)
    omega = rng.randn(d, D) * np.sqrt(2.0 * gamma)   # (d, D)
    b     = rng.uniform(0.0, 2.0 * np.pi, D)         # (D,)
    return omega, b


def compute_rff_features(
    X: np.ndarray,
    omega: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    '''Compute Random Fourier Features for input X.

    z(x) = sqrt(2/D) * cos(omega^T x + b)

    Args:
        X:     Input data of shape (n, d).
        omega: Frequency matrix of shape (d, D).
        b:     Phase offsets of shape (D,).

    Returns:
        RFF feature matrix of shape (n, D).
    '''
    D          = omega.shape[1]
    projection = X @ omega + b          # (n, D)
    return np.sqrt(2.0 / D) * np.cos(projection)


def rff_kernel_approximation(
    X1: np.ndarray,
    X2: np.ndarray,
    D: int = N_RFF,
    gamma: float = GAMMA_RBF,
    random_state: int = SEED,
) -> np.ndarray:
    '''Approximate the RBF kernel matrix using Random Fourier Features.

    Args:
        X1:           Array of shape (n1, d).
        X2:           Array of shape (n2, d).
        D:            Number of random features (more = better approximation).
        gamma:        RBF bandwidth parameter.
        random_state: Random seed.

    Returns:
        Approximate kernel matrix of shape (n1, n2).
    '''
    d           = X1.shape[1]
    omega, b    = sample_rff_weights(d, D, gamma, random_state)
    z1          = compute_rff_features(X1, omega, b)   # (n1, D)
    z2          = compute_rff_features(X2, omega, b)   # (n2, D)
    return z1 @ z2.T                                   # (n1, n2)


# ── RFF approximation quality: exact vs RFF kernel for various D ──────────────
K_exact   = rbf_kernel(X_moons[:100], X_moons[:100], gamma=GAMMA_RBF)
D_values  = [10, 50, 100, 200, 500, 1000, 2000]
rff_errors = []
for D in D_values:
    K_rff = rff_kernel_approximation(X_moons[:100], X_moons[:100],
                                      D=D, gamma=GAMMA_RBF)
    frob_err = float(np.linalg.norm(K_exact - K_rff, 'fro') / np.linalg.norm(K_exact, 'fro'))
    rff_errors.append(frob_err)
    print(f'  D={D:5d}: relative Frobenius error = {frob_err:.6f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.loglog(D_values, rff_errors, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Number of RFF features D')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('RFF Approximation Quality vs D', fontsize=12)

ax = axes[1]
K_rff_best = rff_kernel_approximation(X_moons[:100], X_moons[:100],
                                       D=2000, gamma=GAMMA_RBF)
ax.scatter(K_exact.ravel(), K_rff_best.ravel(), alpha=0.05, s=2, color='tomato')
lim = [K_exact.min(), K_exact.max()]
ax.plot(lim, lim, 'k--', lw=1.5)
ax.set_xlabel('Exact RBF kernel value')
ax.set_ylabel('RFF approximation (D=2000)')
ax.set_title('Exact vs RFF (D=2000)', fontsize=12)

fig.suptitle('Random Fourier Feature Approximation of RBF Kernel', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3 — Training & Application

We now apply our implementations to the benchmark datasets, compare against sklearn, and explore the connection between softmax attention and kernels.

In [ ]:
# ── Kernel PCA on all three datasets with gamma sweep ─────────────────────────
print('Kernel PCA: gamma sensitivity on Moons dataset...')
gamma_vals = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
rows = []
for g in gamma_vals:
    kpca_g = KernelPCA(n_components=2, kernel='rbf', gamma=g)
    Z_g    = kpca_g.fit_transform(X_moons)
    # Separability: between-class / within-class variance ratio
    mu0  = Z_g[y_moons == 0].mean(axis=0)
    mu1  = Z_g[y_moons == 1].mean(axis=0)
    var0 = Z_g[y_moons == 0].var(axis=0).sum()
    var1 = Z_g[y_moons == 1].var(axis=0).sum()
    sep  = float(np.linalg.norm(mu0 - mu1) ** 2 / (var0 + var1 + 1e-10))
    rows.append({'gamma': g, 'EVR_PC1': round(kpca_g.explained_variance_ratio_[0], 4),
                 'EVR_PC2': round(kpca_g.explained_variance_ratio_[1], 4),
                 'class_sep': round(sep, 4)})
    print(f'  gamma={g:.1f}: EVR=[{kpca_g.explained_variance_ratio_[0]:.3f},'
          f'{kpca_g.explained_variance_ratio_[1]:.3f}], class_sep={sep:.4f}')

df_gamma = pd.DataFrame(rows).set_index('gamma')
print('\nGamma sensitivity table:')
print(df_gamma.to_string())

# ── Side-by-side: Linear PCA vs RBF Kernel PCA on Moons ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Linear PCA (via SVD)
U, S, Vt = np.linalg.svd(X_moons - X_moons.mean(axis=0), full_matrices=False)
Z_lin    = U[:, :2] * S[:2]
ax = axes[0]
for cls in [0, 1]:
    mask = y_moons == cls
    ax.scatter(Z_lin[mask, 0], Z_lin[mask, 1], c=[COLORS[cls]], alpha=0.6, s=20)
ax.set_title('Linear PCA (fails to separate)', fontsize=11)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

for ax, (g, title) in zip(axes[1:], [(1.0, 'RBF KPCA (gamma=1.0)'),
                                       (5.0, 'RBF KPCA (gamma=5.0)')]):
    kpca_a = KernelPCA(n_components=2, kernel='rbf', gamma=g)
    Z_a    = kpca_a.fit_transform(X_moons)
    for cls in [0, 1]:
        mask = y_moons == cls
        ax.scatter(Z_a[mask, 0], Z_a[mask, 1], c=[COLORS[cls]], alpha=0.6, s=20)
    ax.set_title(title + '\n(separates clusters)', fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig.suptitle('Linear PCA vs Kernel PCA on Moons Dataset', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

### Gamma Sensitivity Analysis for RBF Kernel PCA

The `gamma` parameter in the RBF kernel controls the width of the Gaussian: small `gamma` → broad kernel (all points similar) → linear-like structure in feature space; large `gamma` → narrow kernel (only neighbours interact) → degenerate projections. We sweep `gamma` and measure separability in the first PC.

In [ ]:
# ── Gamma sensitivity for kernel PCA on Circles ────────────────────────────
print("=== RBF Kernel PCA: Gamma Sensitivity Analysis ===\n")

X_circ_g, y_circ_g = make_circles(n_samples=N_SAMPLES, noise=0.08,
                                   factor=0.5, random_state=SEED)
X_circ_g = StandardScaler().fit_transform(X_circ_g)

gammas_list: list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
gamma_rows: list = []

for g in gammas_list:
    kpca_g = KernelPCA(n_components=2, kernel='rbf', gamma=g)
    Z_g = kpca_g.fit_transform(X_circ_g)
    ev_ratio = kpca_g.explained_variance_ratio_[:2]
    # Between-class distance in PC1 as separability proxy
    mu0_pc1: float = float(Z_g[y_circ_g == 0, 0].mean())
    mu1_pc1: float = float(Z_g[y_circ_g == 1, 0].mean())
    between_var: float = (mu0_pc1 - mu1_pc1) ** 2
    total_var: float = float(Z_g[:, 0].var()) + 1e-10
    sep_score: float = between_var / total_var
    gamma_rows.append({
        'gamma': g,
        'PC1_var_ratio': float(ev_ratio[0]),
        'PC2_var_ratio': float(ev_ratio[1]),
        'separability': sep_score,
    })
    print(f'gamma={g:5.2f} | PC1_var={ev_ratio[0]:.3f} | PC2_var={ev_ratio[1]:.3f} | sep={sep_score:.3f}')

df_gamma = pd.DataFrame(gamma_rows)
best_gamma_val: float = float(df_gamma.loc[df_gamma['separability'].idxmax(), 'gamma'])
print(f'\nBest gamma (highest separability): {best_gamma_val}')

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for idx, (g, ax) in enumerate(zip(gammas_list, axes.ravel())):
    kpca_g = KernelPCA(n_components=2, kernel='rbf', gamma=g)
    Z_g = kpca_g.fit_transform(X_circ_g)
    ax.scatter(Z_g[y_circ_g == 0, 0], Z_g[y_circ_g == 0, 1],
               s=12, alpha=0.7, c='steelblue', label='Inner')
    ax.scatter(Z_g[y_circ_g == 1, 0], Z_g[y_circ_g == 1, 1],
               s=12, alpha=0.7, c='coral', label='Outer')
    sep_val = df_gamma.loc[idx, 'separability']
    title_col = 'darkgreen' if g == best_gamma_val else 'black'
    ax.set_title(f'gamma={g}  sep={sep_val:.2f}', color=title_col, fontsize=9)
    ax.set_xlabel('PC 1', fontsize=8)
    ax.set_ylabel('PC 2', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('Kernel PCA Projections — Varying RBF gamma (Circles Dataset)', fontsize=12)
plt.tight_layout()
plt.savefig('kpca_gamma_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nSmall gamma: kernel nearly constant => all points look alike => no separation.')
print('Large gamma: kernel ~ delta => each point isolated => overfits, poor generalisation.')

### Softmax Attention as a Kernel

Scaled dot-product attention can be viewed as a kernel method where the attention matrix is an *asymmetric* kernel:
$$A_{ij} = \frac{\exp(\mathbf{q}_i^\top \mathbf{k}_j / \sqrt{d})}{\sum_\ell \exp(\mathbf{q}_i^\top \mathbf{k}_\ell / \sqrt{d})}$$

This is equivalent to using the (normalised) **exponential kernel** $k(\mathbf{q}, \mathbf{k}) \propto \exp(\mathbf{q}^\top \mathbf{k} / \sqrt{d})$. The kernel perspective provides an alternative way to understand efficient attention (Performer, Linear Transformer) via random feature approximations of the exponential kernel.

In [ ]:
def softmax_attention_weights(
    Q: np.ndarray,
    K: np.ndarray,
    temperature: float = 1.0,
) -> np.ndarray:
    '''Compute scaled dot-product attention weight matrix.

    A[i, j] = softmax_j(q_i^T k_j / (sqrt(d) * temperature))

    This is an asymmetric kernel matrix: each row is a probability
    distribution over the key positions.

    Args:
        Q:           Query matrix of shape (n_q, d).
        K:           Key matrix of shape (n_k, d).
        temperature: Softmax temperature (1.0 = standard scaling by sqrt(d)).

    Returns:
        Attention weight matrix of shape (n_q, n_k).
    '''
    d      = Q.shape[1]
    scores = Q @ K.T / (np.sqrt(d) * temperature)  # (n_q, n_k)
    # Numerically stable row-wise softmax
    scores -= scores.max(axis=1, keepdims=True)
    exp_s  = np.exp(scores)
    return exp_s / exp_s.sum(axis=1, keepdims=True)


def dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    temperature: float = 1.0,
) -> np.ndarray:
    '''Scaled dot-product attention: output = softmax(QK^T/sqrt(d)) V.

    Args:
        Q:           Query matrix of shape (n_q, d_k).
        K:           Key matrix of shape (n_k, d_k).
        V:           Value matrix of shape (n_k, d_v).
        temperature: Softmax temperature.

    Returns:
        Output matrix of shape (n_q, d_v).
    '''
    A = softmax_attention_weights(Q, K, temperature=temperature)
    return A @ V


# ── Demo: attention as kernel interpolation ───────────────────────────────────
rng_att   = np.random.RandomState(SEED)
n_q, n_k, d_k, d_v = 8, 12, 4, 2
Q_demo = rng_att.randn(n_q, d_k)
K_demo = rng_att.randn(n_k, d_k)
V_demo = rng_att.randn(n_k, d_v)

A_demo  = softmax_attention_weights(Q_demo, K_demo)
out_att = dot_product_attention(Q_demo, K_demo, V_demo)

print(f'Attention matrix shape : {A_demo.shape}')
print(f'Output shape           : {out_att.shape}')
print(f'Row sums (should be 1): {np.round(A_demo.sum(axis=1), 6)}')

# ── Temperature effect on attention sharpness ─────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, tau in zip(axes, [0.1, 0.5, 1.0, 5.0]):
    A_tau = softmax_attention_weights(Q_demo, K_demo, temperature=tau)
    im = ax.imshow(A_tau, cmap='Blues', vmin=0, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.85)
    entropy = float(-np.sum(A_tau * np.log(A_tau + 1e-12)) / n_q)
    ax.set_title(f'T={tau}\nmean entropy={entropy:.3f}', fontsize=10)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
fig.suptitle('Softmax Attention (kernel view) — Temperature Effect',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Kernel vs attention on same data ─────────────────────────────────────────
X_small = X_moons[:20]
K_rbf_small   = rbf_kernel(X_small, X_small, gamma=1.0)           # symmetric
A_att_small   = softmax_attention_weights(X_small, X_small)       # asymmetric
print(f'\nRBF kernel symmetry error:  {np.max(np.abs(K_rbf_small - K_rbf_small.T)):.2e}')
print(f'Attention symmetry error:   {np.max(np.abs(A_att_small - A_att_small.T)):.2e}')
print('-> RBF is symmetric (valid Mercer kernel); attention is NOT.')

---
## Part 4 — Evaluation & Analysis

We run ablation studies across key hyperparameters and compare our implementations against sklearn references.

In [ ]:
# ── sklearn KernelPCA comparison table ───────────────────────────────────────
print('Kernel PCA comparison: our implementation vs sklearn')
print('-' * 60)
comp_rows = []
for dataset_name, X_d, y_d in datasets:
    for kname, kargs_ours, kargs_skl in [
        ('rbf',    {'gamma': 1.0},    {'gamma': 1.0}),
        ('poly',   {'degree': 3},     {'degree': 3, 'coef0': 1.0}),
        ('linear', {},                {}),
    ]:
        # Our KernelPCA
        kp_ours = KernelPCA(n_components=2, kernel=kname, **kargs_ours)
        Z_ours  = kp_ours.fit_transform(X_d)

        # sklearn KernelPCA
        kp_skl  = SklearnKernelPCA(n_components=2, kernel=kname, **kargs_skl)
        Z_skl   = kp_skl.fit_transform(X_d)

        # Correlation (sign-invariant)
        corr = min(abs(float(np.corrcoef(Z_ours[:, c], Z_skl[:, c])[0, 1]))
                   for c in range(2))
        comp_rows.append({
            'Dataset': dataset_name,
            'Kernel':  kname,
            'Min corr vs sklearn': round(corr, 5),
            'Match': 'YES' if corr > 0.999 else 'NO',
        })
        print(f'  {dataset_name:7s} {kname:6s}: corr={corr:.5f}')

df_comp = pd.DataFrame(comp_rows)
print('\nAll matches (corr > 0.999):', all(df_comp['Match'] == 'YES'))

# ── sklearn KDE comparison ────────────────────────────────────────────────────
print('\nKDE comparison (log-density at training points):')
h_cmp = silverman_bandwidth(X_moons)

# Our implementation
log_d_ours = gaussian_kde_log_density(X_moons, X_moons[:10], bandwidth=h_cmp)

# sklearn
skl_kde = SklearnKDE(bandwidth=h_cmp, kernel='gaussian')
skl_kde.fit(X_moons)
log_d_skl = skl_kde.score_samples(X_moons[:10])

max_kde_err = float(np.max(np.abs(log_d_ours - log_d_skl)))
print(f'  Max abs error vs sklearn KDE: {max_kde_err:.6f}')
assert max_kde_err < 1e-6, 'KDE log-density deviates from sklearn!'
print('  KDE implementation matches sklearn.')

In [ ]:
def fit_kernel_ridge(
    K_train: np.ndarray,
    y_train: np.ndarray,
    alpha: float = 0.01,
) -> np.ndarray:
    '''Fit kernel ridge regression by solving the dual problem.

    Minimises ||K c - y||^2 + alpha * c^T K c  =>  (K + alpha I) c = y.

    Args:
        K_train: Training kernel matrix of shape (n, n).
        y_train: Target values of shape (n,).
        alpha:   Regularisation strength.

    Returns:
        Dual coefficients c of shape (n,).
    '''
    n = K_train.shape[0]
    A = K_train + alpha * np.eye(n)
    return np.linalg.solve(A, y_train)


def predict_kernel_ridge(
    K_test: np.ndarray,
    c: np.ndarray,
) -> np.ndarray:
    '''Predict using kernel ridge regression dual coefficients.

    Args:
        K_test: Test kernel matrix of shape (n_test, n_train).
        c:      Dual coefficients of shape (n_train,).

    Returns:
        Predictions of shape (n_test,).
    '''
    return K_test @ c


# ── Kernel ridge regression on Circles dataset (regression target = radius) ───
# Regression target: squared radius (0 = inner ring, ~1 = outer ring)
y_reg = X_circles[:, 0] ** 2 + X_circles[:, 1] ** 2

# Train/test split (manual)
X_tr, X_te, y_tr, y_te = tts(X_circles, y_reg, test_size=0.2, random_state=SEED)

# RBF kernel ridge
K_tr = rbf_kernel(X_tr, X_tr, gamma=GAMMA_RBF)
K_te = rbf_kernel(X_te, X_tr, gamma=GAMMA_RBF)

alphas_to_try = [0.001, 0.01, 0.1, 1.0, 10.0]
results_krr   = []
for a in alphas_to_try:
    c       = fit_kernel_ridge(K_tr, y_tr, alpha=a)
    y_pred  = predict_kernel_ridge(K_te, c)
    mse     = float(np.mean((y_pred - y_te) ** 2))
    r2      = float(1 - np.sum((y_pred - y_te)**2) / np.sum((y_te - y_te.mean())**2))
    results_krr.append({'alpha': a, 'MSE': round(mse, 5), 'R2': round(r2, 4)})
    print(f'  alpha={a:.3f}: MSE={mse:.5f}, R2={r2:.4f}')

best_a    = min(results_krr, key=lambda r: r['MSE'])['alpha']
c_best    = fit_kernel_ridge(K_tr, y_tr, alpha=best_a)
y_pred_best = predict_kernel_ridge(K_te, c_best)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
sc = ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='viridis', s=40)
plt.colorbar(sc, ax=ax, label='True radius^2')
ax.set_title('Kernel Ridge Regression: True Target', fontsize=11)
ax.set_xlabel('F1')
ax.set_ylabel('F2')

ax = axes[1]
sc = ax.scatter(X_te[:, 0], X_te[:, 1], c=y_pred_best, cmap='viridis', s=40)
plt.colorbar(sc, ax=ax, label='Predicted')
ax.set_title(f'KRR Prediction (alpha={best_a}, R2={r2:.3f})', fontsize=11)
ax.set_xlabel('F1')
plt.tight_layout()
plt.show()

In [ ]:
# ── RFF approximation quality: varying gamma ──────────────────────────────────
print('RFF approximation quality vs gamma (D=500):')
gamma_rff_vals = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
X_rff_test     = X_moons[:80]
rff_gamma_rows = []

for g in gamma_rff_vals:
    K_ex  = rbf_kernel(X_rff_test, X_rff_test, gamma=g)
    K_rff = rff_kernel_approximation(X_rff_test, X_rff_test,
                                      D=500, gamma=g, random_state=SEED)
    err   = float(np.linalg.norm(K_ex - K_rff, 'fro') / np.linalg.norm(K_ex, 'fro'))
    rff_gamma_rows.append({'gamma': g, 'rel_error': round(err, 5)})
    print(f'  gamma={g:.1f}: relative error={err:.5f}')

df_rff_gamma = pd.DataFrame(rff_gamma_rows).set_index('gamma')

# ── Kernel ablation: which kernel best separates Moons and Circles ────────────
print('\nClass separability after KernelPCA (var-ratio metric):')
abl_rows = []
for ds_name, X_d, y_d in datasets:
    for kname, kparams in [
        ('linear', {}),
        ('rbf-g0.5', {'kernel': 'rbf', 'gamma': 0.5}),
        ('rbf-g1',   {'kernel': 'rbf', 'gamma': 1.0}),
        ('rbf-g5',   {'kernel': 'rbf', 'gamma': 5.0}),
        ('poly-d2',  {'kernel': 'poly', 'degree': 2}),
        ('poly-d3',  {'kernel': 'poly', 'degree': 3}),
    ]:
        k_type = kparams.pop('kernel', kname.split('-')[0])
        kpca_a = KernelPCA(n_components=2, kernel=k_type, **kparams)
        Z      = kpca_a.fit_transform(X_d)
        # Between-class / within-class variance
        classes = np.unique(y_d)
        class_means  = np.stack([Z[y_d == c].mean(axis=0) for c in classes])
        global_mean  = Z.mean(axis=0)
        btwn_var = float(np.sum([(y_d == c).sum() *
                                  np.linalg.norm(class_means[c] - global_mean)**2
                                  for c in classes]) / len(X_d))
        within_var = float(np.sum([
            np.sum((Z[y_d == c] - class_means[c])**2)
            for c in classes
        ]) / len(X_d))
        sep_ratio = btwn_var / (within_var + 1e-10)
        abl_rows.append({'Dataset': ds_name, 'Kernel': kname,
                          'Sep_ratio': round(sep_ratio, 4)})
        print(f'  {ds_name:8s} {kname:10s}: sep_ratio={sep_ratio:.4f}')

df_abl = pd.DataFrame(abl_rows).pivot(index='Kernel', columns='Dataset',
                                       values='Sep_ratio')
print('\nSeparability ratio table (higher = better separated):')
print(df_abl.to_string())

In [ ]:
# ── Comprehensive kernel × dataset accuracy comparison ─────────────────────
print("=== Kernel SVM Accuracy: Kernels × Datasets ===\n")


# Three canonical datasets used in this notebook
_ds_catalog: dict = {
    'Moons':   make_moons(n_samples=300, noise=0.12, random_state=SEED),
    'Circles': make_circles(n_samples=300, noise=0.08, factor=0.5, random_state=SEED),
    'Blobs':   make_blobs(n_samples=300, centers=3, cluster_std=0.9, random_state=SEED),
}
_kernel_names: list = ['linear', 'poly', 'rbf', 'sigmoid']
_cmp_rows: list = []

for ds_name, (X_ds, y_ds) in _ds_catalog.items():
    X_ds_sc = StandardScaler().fit_transform(X_ds)
    for kname in _kernel_names:
        svc_cv = SVC(kernel=kname, C=1.0, gamma='scale', random_state=SEED)
        acc_scores = _cv_score(svc_cv, X_ds_sc, y_ds, cv=5, scoring='accuracy')
        mean_acc: float = float(acc_scores.mean())
        std_acc: float = float(acc_scores.std())
        _cmp_rows.append({
            'Dataset': ds_name,
            'Kernel': kname,
            'Mean Acc': mean_acc,
            'Std Acc': std_acc,
        })
        print(f'{ds_name:8s} | {kname:8s} | acc={mean_acc:.3f} +/- {std_acc:.3f}')

df_cmp = pd.DataFrame(_cmp_rows)
pivot_cmp = df_cmp.pivot(index='Dataset', columns='Kernel', values='Mean Acc')
print('\nSeparability matrix (5-fold CV accuracy):')
print(pivot_cmp.round(3).to_string())

# Visualise as heatmap + bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax_heat = axes[0]
im = ax_heat.imshow(pivot_cmp.values, cmap='YlOrRd', vmin=0.5, vmax=1.0)
ax_heat.set_xticks(range(len(pivot_cmp.columns)))
ax_heat.set_xticklabels(pivot_cmp.columns, fontsize=10)
ax_heat.set_yticks(range(len(pivot_cmp.index)))
ax_heat.set_yticklabels(pivot_cmp.index, fontsize=10)
for ri in range(len(pivot_cmp.index)):
    for ci in range(len(pivot_cmp.columns)):
        cell_val = pivot_cmp.values[ri, ci]
        txt_col = 'white' if cell_val > 0.85 else 'black'
        ax_heat.text(ci, ri, f'{cell_val:.3f}',
                     ha='center', va='center', fontsize=11, color=txt_col)
plt.colorbar(im, ax=ax_heat, label='5-fold CV Accuracy')
ax_heat.set_title('Kernel x Dataset Accuracy Heatmap')

ax_bar = axes[1]
x_pos = np.arange(len(_ds_catalog))
width = 0.2
bar_colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
for ki, kname in enumerate(_kernel_names):
    vals = [pivot_cmp.loc[ds, kname] for ds in _ds_catalog.keys()]
    ax_bar.bar(x_pos + ki * width, vals, width=width,
               color=bar_colors[ki], label=kname, alpha=0.85)
ax_bar.set_xticks(x_pos + 1.5 * width)
ax_bar.set_xticklabels(list(_ds_catalog.keys()))
ax_bar.set_ylim(0.4, 1.05)
ax_bar.set_ylabel('5-fold CV Accuracy')
ax_bar.set_title('Kernel Accuracy by Dataset')
ax_bar.legend(fontsize=9)
ax_bar.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('kernel_dataset_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nKey insight: RBF is safest default; linear SVM often matches on linearly-separable data.')
print('No single kernel wins on all datasets — match kernel to geometry, not habit.')

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **The kernel trick trades computation for expressiveness.** Instead of explicitly mapping to a high-dimensional feature space (intractable for RBF), we compute inner products $k(\mathbf{x}, \mathbf{z})$ directly — $O(n^2 d)$ regardless of feature dimension.

2. **Mercer's theorem is the validity criterion.** A kernel is valid if its Gram matrix is PSD for all datasets. RBF and polynomial kernels always satisfy this; the sigmoid kernel only does so for specific parameter ranges.

3. **Kernel PCA reveals non-linear structure.** By centering in the feature space and finding the top eigenvectors of the kernel matrix, we separate clusters that PCA cannot. The RBF bandwidth $\gamma$ controls locality — large $\gamma$ sees only nearby structure.

4. **Random Fourier Features bridge kernels and linear methods.** Sampling $D$ Fourier features approximates the RBF kernel as $z(\mathbf{x})^\top z(\mathbf{z})$, reducing kernel methods from $O(n^2)$ to $O(nD)$ — the approach used in Performer attention and other efficient transformers.

5. **Softmax attention is an asymmetric kernel.** The attention matrix $A_{ij} \propto \exp(\mathbf{q}_i^\top \mathbf{k}_j / \sqrt{d})$ uses the exponential kernel but normalises per query, breaking symmetry. This connection motivates kernel-based approximations of attention.

### What's Next

→ **03-09 (Matrix Factorization)** builds on the SVD and eigendecomposition mechanics we used in kernel PCA to implement NMF and truncated SVD.

→ **08-02 (Multi-Head Attention)** revisits the attention-as-kernel connection with full transformer context.

### Going Further

- Rahimi & Recht (2007) *Random Features for Large-Scale Kernel Machines* — RFF original paper.
- Schölkopf, Smola & Müller (1998) *Nonlinear Component Analysis as a Kernel Eigenvalue Problem* — kernel PCA.
- Choromanski et al. (2021) *Rethinking Attention with Performers* — RFF approximation of attention.